# PROJECT 4.2 HEART DISEASE
## `BEST MODEL SELECTION`

## 1. LOAD DATASET

In [30]:
import warnings
warnings.filterwarnings("ignore")
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import StratifiedKFold
from sklearn.tree import DecisionTreeClassifier

from src.data.data_engineering import data_loader

path = "../dataset/"
X_prp_train, X_prp_val, X_prp_test, y_prp_train, y_prp_val, y_prp_test = data_loader(path, filenames = ("prp_train.csv", "prp_val.csv", "prp_test.csv"))
X_odt_train, X_odt_val, X_odt_test, y_odt_train, y_odt_val, y_odt_test = data_loader(path, filenames=("topK_odt_train.csv", "topK_odt_val.csv", "topK_odt_test.csv"))
X_fe_dt_train, X_fe_dt_val, X_fe_dt_test, y_fe_dt_train, y_fe_dt_val, y_fe_dt_test = data_loader(path, filenames=("topK_fe_dt_train.csv", "topK_fe_dt_val.csv", "topK_fe_dt_test.csv"))

✅ Load data from ('prp_train.csv', 'prp_val.csv', 'prp_test.csv') successfully
✅ Load data from ('topK_odt_train.csv', 'topK_odt_val.csv', 'topK_odt_test.csv') successfully
✅ Load data from ('topK_fe_dt_train.csv', 'topK_fe_dt_val.csv', 'topK_fe_dt_test.csv') successfully


In [10]:
X_fe_dt_train.head(10)

,oldpeak,chol_per_age,chol,thal_3.0,trestbps,age,ca_0.0,exang_1.0,cp_1.0,sex_0.0,bps_per_age,hr_ratio,slope_2.0,thalach,ca_2.0
0,1.320132,-0.313670,0.169014,0.0,1.020911,0.594945,0.0,0.0,0.0,1.0,0.062907,-0.301337,1.0,0.307680,1.0
1,-0.900796,3.088780,1.327117,1.0,0.463338,-1.711319,1.0,0.0,0.0,0.0,2.364493,2.172472,0.0,1.411752,0.0
2,-0.900796,0.580777,0.720492,1.0,0.017280,-0.063987,0.0,1.0,0.0,1.0,-0.054429,0.094204,0.0,0.396006,0.0
3,-0.900796,-0.619719,-0.511143,0.0,-0.094234,0.155657,1.0,0.0,0.0,0.0,-0.314244,0.053501,0.0,0.572657,0.0
4,-0.729955,-0.754219,0.095483,0.0,-0.373021,1.363700,0.0,0.0,0.0,0.0,-1.276822,-0.523173,1.0,0.572657,1.0
5,-0.900796,0.087651,0.242544,1.0,-0.094234,0.045835,1.0,0.0,0.0,0.0,-0.225099,-0.058162,0.0,0.219354,0.0
6,-0.900796,0.623750,0.077101,1.0,-0.094234,-0.832742,1.0,0.0,0.0,0.0,0.624620,1.136904,0.0,1.279263,0.0
7,1.490973,-0.313670,0.169014,0.0,-0.373021,0.594945,0.0,1.0,0.0,0.0,-0.817111,-0.623134,1.0,-0.398926,0.0
8,-0.046593,-0.334564,-0.014812,1.0,-1.766952,0.375301,1.0,0.0,0.0,1.0,-1.575748,-0.920656,1.0,-1.238020,0.0
9,2.003495,0.747049,1.419030,0.0,2.136056,0.485123,1.0,1.0,0.0,0.0,0.868347,-0.595522,0.0,-0.443089,0.0


## 2.TRAINING THE ADABOOST MODEL

In [38]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import GridSearchCV
from sklearn.utils.class_weight import compute_sample_weight # Tính toán trọng số trong trường hợp mất cân bằng lớp

### Try not to use iterator, the method is Decision Classifier and Kfold

In [39]:
def find_optimum_abd(X_train, y_train,
                     n_splits=5
                     ):
    """
    function to find the optimum Adaboost model based on the training data
    :return: Adaboost model, accuracy score
    """
    print("👉 Finding optimum Adaboost model...")
    param_grid = {
    'n_estimators': [50, 100, 200, 300],
    'learning_rate': [0.01, 0.1, 0.5, 1.0, 1.5],
    'estimator__max_depth': [1, 2, 3, 4],
    'algorithm': ['SAMME', 'SAMME.R']
    }

    # Create base estimator
    base_estimator = DecisionTreeClassifier(random_state=42)

    # Adaboost estimator
    ada = AdaBoostClassifier(estimator=base_estimator, random_state=42) # Remove the learning_rate from this state
    print(f"estimator: {ada.estimator}")

    # Create Strafified Kfold
    folds = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

    # Setup StratifiedKfold
    grid_search = GridSearchCV(
        estimator=ada,
        param_grid=param_grid,
        cv=folds,
        scoring='accuracy',
        n_jobs=-1,
        verbose=1
    )

    # Calculate sample weight for unbalanced data
    sw = compute_sample_weight('balanced', y=y_train)

    grid_search.fit(X_train, y_train, sample_weight=sw)
    print(f"✅ Best parameters: {grid_search.best_params_}")
    print(f"✅ Best CV score: {grid_search.best_score_:.4f}")

    return grid_search.best_estimator_, grid_search.best_params_


In [40]:
best_ada_fe, best_ada_parameters = find_optimum_abd(X_fe_dt_train, y_fe_dt_train)
print("Best estimator: ", best_ada_parameters)

👉 Finding optimum Adaboost model...
estimator: DecisionTreeClassifier(random_state=42)
Fitting 5 folds for each of 160 candidates, totalling 800 fits
✅ Best parameters: {'algorithm': 'SAMME', 'estimator__max_depth': 1, 'learning_rate': 1.5, 'n_estimators': 300}
✅ Best CV score: 0.3981
Best estimator:  {'algorithm': 'SAMME', 'estimator__max_depth': 1, 'learning_rate': 1.5, 'n_estimators': 300}


In [42]:
y_pred = best_ada_fe.predict(X_fe_dt_test)
accuracy_score(y_pred, y_fe_dt_test)

0.5161290322580645

In [43]:
best_ada_odt, _ = find_optimum_abd(X_odt_train, y_odt_train)
y_odt_pred = best_ada_odt.predict(X_odt_test)
accuracy_score(y_pred, y_odt_test)

👉 Finding optimum Adaboost model...
estimator: DecisionTreeClassifier(random_state=42)
Fitting 5 folds for each of 160 candidates, totalling 800 fits
✅ Best parameters: {'algorithm': 'SAMME', 'estimator__max_depth': 1, 'learning_rate': 1.0, 'n_estimators': 200}
✅ Best CV score: 0.4120


0.5161290322580645